Project: Fine-Tune CNN Model for Age Detection (UTKFace Dataset)

🧠 Model Choice: We'll use MobileNetV2 (lightweight and effective for transfer learning).
Optionally, you can swap it with DeepFace, ResNet50, etc.

📁 Step 1: Dataset Preparation
UTKFace filenames are in this format: age_gender_race_date.jpg

In [3]:
#!pip install scikit-learn
#!pip install numpy
#!pip install opencv-python
#!pip install tensorflow
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Load the dataset

def load_utkface_data(dataset_path , image_size=(100, 100), max_images=8000):
    X, y = [], []
    for i, filename in enumerate(os.listdir(dataset_path)):
        if i >= max_images:
            break
        try:
            age = int(filename.split('_')[0])
            img_path = os.path.join(dataset_path, filename)
            img = cv2.imread(img_path)
            img = cv2.resize(img, image_size)
            X.append(img)
            y.append(age)
        except:
            continue
    return np.array(X), np.array(y)

X, y = load_utkface_data("C:\\Users\\SHWETA BHOYAR\\OneDrive\\CAPM WEB APPLICATION\\UTKFace")
X = X / 255.0  # normalize

# You can categorize ages or treat it as regression
# Option 1: Regression
y_reg = y

# Option 2: Classification (age bins, e.g., 0–10, 11–20, etc.)
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 100]
y_class = np.digitize(y, bins)
y_cat = to_categorical(y_class)

X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=42)


🏗️ Step 2: Build and Fine-Tune the Model

In [4]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(200, 200, 3))
base_model.trainable = False  # Freeze base

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(len(bins)+1, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


C:\Users\SHWETA BHOYAR\AppData\Local\Temp\ipykernel_22964\4244355303.py:6: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(200, 200, 3))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 200, 200,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 100, 100,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 100, 100,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 100, 100,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 100, 100,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 100, 100,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 100, 100,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 100, 100,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 100, 100,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 100, 100,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 100, 100,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 100, 100,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 101, 101,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 50, 50,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 50, 50,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 50, 50,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 50, 50,    │      2,304 │ block_1_depthwis

 Total params: 2,423,371 (9.24 MB)

 Trainable params: 165,387 (646.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [5]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)


(6400, 100, 100, 3)
(6400, 11)
(1600, 100, 100, 3)
(1600, 11)


In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, InputLayer

model = Sequential()
model.add(InputLayer(input_shape=(100, 100, 3)))  # Must match X_train shape


c:\anaconda\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


🏋️ Step 3: Train the Model

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential()

# Input shape should match (100, 100, 3)
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(100, 100, 3)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(11, activation='softmax'))  # Because y_train has shape (N, 11)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])



c:\anaconda\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
history = model.fit(X_train, y_train, 
                    validation_data=(X_test, y_test),
                    epochs=10,
                    batch_size=32)


Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 44s 199ms/step - accuracy: 0.6764 - loss: 0.9057 - val_accuracy: 0.8225 - val_loss: 0.4671
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 37s 185ms/step - accuracy: 0.8224 - loss: 0.4610 - val_accuracy: 0.8363 - val_loss: 0.4402
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 34s 170ms/step - accuracy: 0.8372 - loss: 0.4171 - val_accuracy: 0.8306 - val_loss: 0.4445
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 36s 179ms/step - accuracy: 0.8490 - loss: 0.3904 - val_accuracy: 0.8294 - val_loss: 0.5073
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 35s 175ms/step - accuracy: 0.8691 - loss: 0.3455 - val_accuracy: 0.8562 - val_loss: 0.3852
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 37s 183ms/step - accuracy: 0.8753 - loss: 0.2983 - val_accuracy: 0.8531 - val_loss: 0.4214
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 36s 178ms/step - accuracy: 0.8915 - loss: 0.2764 - val_accuracy: 0.8644 - val_loss: 0.4078
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 37s 185ms/step - accuracy: 0.8939 - loss: 0

📊 Step 4: Evaluate the Accuracy

In [9]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.8439 - loss: 0.5498
Test Accuracy: 85.37%


📊 Next Steps (Polish & Package for Submission)
Here’s what you should do now, step-by-step:

✅ 1. Evaluate the Model (Precision, Recall, Confusion Matrix)
Paste this code below your training cell to generate evaluation metrics:

In [10]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Predict on test data
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred_classes))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes))


50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step
Confusion Matrix:
[[ 215    3   11    0]
 [   3  129  195    0]
 [   1   15 1022    0]
 [   0    2    4    0]]

Classification Report:
              precision    recall  f1-score   support

           1       0.98      0.94      0.96       229
           2       0.87      0.39      0.54       327
           3       0.83      0.98      0.90      1038
          10       0.00      0.00      0.00         6

    accuracy                           0.85      1600
   macro avg       0.67      0.58      0.60      1600
weighted avg       0.86      0.85      0.83      1600



c:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✅ 2. Save Your Model

In [11]:
model.save("age_detection_model.h5")

In [15]:
model = load_model("age_detection_model.h5")
